# App Reviews Data Preparation
## Cleaning and Labeling Google Play Store Reviews for Domain Shift Testing

**Project Team: Group K**
**Date:** June 2026

---

## 1. Problem Definition & Introduction

### Goal
To prepare a brand new test set using Google App Reviews. This dataset represents a **Domain Shift** from the Twitter training data, as it contains structured product feedback rather than general social media posts.

### Why App Reviews?
App reviews are highly opinionated and contain clear sentiment signals (emotional words, specific complaints, or praise). This makes them an excellent target for testing how well our models generalize to "Product Review" domains.

## 2. Importing the necessary Libraries
> **Note:** These are the same libraries used in the main Sentiment Analysis notebook to ensure consistency.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="pastel")

## 3. Data Loading
Loading the raw app reviews gathered for this project.

In [ ]:
raw_data_path = '../data/raw/app_reviews_raw.csv'
df_app = pd.read_csv(raw_data_path)
print(f"Loaded {len(df_app)} raw reviews.")

## 4. Exploratory Data Analysis (EDA)
Following the same EDA standards as the main project notebook for the test datasets.

### 4.1 Viewing the first few rows (App Reviews)

In [ ]:
display("--- App Reviews Dataset ---")
display(df_app.head(3))

### 4.2 Dataset Summaries (Info & Describe)

In [ ]:
print("=== INFO: app_reviews_raw.csv ===")
df_app.info()
print("\n=== DESCRIBE: app_reviews_raw.csv ===")
display(df_app.describe(include='all'))

### 4.3 Checking for null values

In [ ]:
print("\n=== APP REVIEWS NULL VALUES CHECK ===")
print(df_app.isnull().sum())

### 4.4 Checking the Rating Distribution
We use the `score` column (1-5) as a proxy for the initial class distribution.

In [ ]:
print("App Reviews Score Distribution:")
print(df_app.score.value_counts(dropna=False))

### 4.5 Visualising the Score Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='score', data=df_app, palette='viridis', edgecolor='black')
plt.title('App Reviews — Score Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Score (1-5)', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)

for p in plt.gca().patches:
    plt.gca().annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.show()

### 4.6 Text Length Analysis
Analyzing how long the reviews are. This helps us compare the domain structure to Twitter.

In [ ]:
df_app['text_length'] = df_app['content'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(df_app['text_length'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of Review Word Counts', fontsize=14, fontweight='bold')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.xlim(0, 150)
plt.show()

## 5. Data Cleaning And Preprocessing
### 5.1 Text Cleaning Pipeline
We use the exact same cleaning logic as the main notebook to ensure the data format matches the training data.

In [ ]:
def clean_the_text(text):
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove specific noise characters (!!!, ???, @user)
    text = re.sub(r'!!!|\?\?\?|@user', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df_app['clean_text'] = df_app['content'].apply(clean_the_text)
df_app = df_app.drop_duplicates(subset=['clean_text']).reset_index(drop=True)
print(f"Records after cleaning and deduplication: {len(df_app)}")

## 6. Sentiment Labeling Strategy
### 6.1 Mapping Scores to Sentiment Classes
Mapping the 1-5 scores to Positive, Neutral, and Negative categories.

In [ ]:
def map_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df_app['sentiment'] = df_app['score'].apply(map_sentiment)
display(df_app[['content', 'score', 'sentiment']].head(10))

### 6.2 Visualising the New Class Distribution
Showing the distribution after mapping scores to sentiment labels.

In [ ]:
sentiment_counts = df_app['sentiment'].value_counts()
plt.figure(figsize=(10, 6))
colors = ['#4CAF50', '#F44336', '#2196F3'] # Pos, Neg, Neu
sentiment_counts.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Mapped Sentiment Distribution (App Reviews)', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

### 6.3 Manual Verification Prompt
> **Instruction for David:** Madam mentioned labeling the data yourselves. While the scores provide a great baseline, you should manually review a sample (e.g., 200 reviews) to ensure the `sentiment` column accurately reflects the `content`.

## 7. Saving the Processed Test Set
Saving the final cleaned and labeled dataset to the processed data folder.

In [ ]:
output_path = '../data/processed/app_reviews_test.csv'
df_app[['clean_text', 'sentiment']].to_csv(output_path, index=False)
print(f"Success! Cleaned App Reviews test set saved to: {output_path}")